# Currency Exchange Rate Forecasting — EDA

**Project:** Forecasting Currency Exchange Rates for Banking Operations  
**Dataset:** Historical OHLCV FX data (EUR/USD, GBP/USD, USD/JPY, etc.)  
**Objective:** Explore price dynamics, seasonality, volatility, and feature relationships
before model training.

---

## Table of Contents
1. Setup and Imports
2. Data Loading
3. Price Series Analysis
4. Returns Distribution
5. Volatility Analysis
6. Technical Indicators
7. Correlation Analysis
8. Stationarity Tests
9. Summary and Next Steps

In [ ]:
# Cell 1: Setup and Imports
import sys
import warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from src.data_preprocessing import (
    generate_synthetic_forex_data,
    clean_forex_data,
    add_technical_indicators,
    add_lag_features,
    add_calendar_features,
    train_test_split_time_series,
    CURRENCY_PAIRS
)

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100
print('Setup complete.')
print('Currency pairs:', CURRENCY_PAIRS[:5])

In [ ]:
# Cell 2: Data Loading (using synthetic data for demo)
pairs_to_analyze = ['EURUSD=X', 'GBPUSD=X', 'USDJPY=X']
dfs = {}

for pair in pairs_to_analyze:
    raw = generate_synthetic_forex_data(pair=pair, n_days=2000, seed=hash(pair) % 1000)
    df = clean_forex_data(raw)
    df = add_technical_indicators(df)
    df = add_calendar_features(df)
    df = df.dropna()
    dfs[pair] = df
    print(f'{pair}: {len(df)} rows, {df.shape[1]} features')

main_df = dfs['EURUSD=X']
print('\nFeature columns (sample):')
print(main_df.columns.tolist()[:20])

In [ ]:
# Cell 3: Price Series Analysis
fig = make_subplots(rows=2, cols=1,
                    subplot_titles=('EUR/USD Close Price with Bollinger Bands',
                                   'GBP/USD vs USD/JPY (Normalized)'),
                    row_heights=[0.6, 0.4])

# EUR/USD with Bollinger Bands
fig.add_trace(go.Scatter(x=main_df.index, y=main_df['Close'],
                         name='EUR/USD Close', line=dict(color='royalblue')), row=1, col=1)
if 'bb_upper' in main_df.columns:
    fig.add_trace(go.Scatter(x=main_df.index, y=main_df['bb_upper'],
                             name='BB Upper', line=dict(color='gray', dash='dot')), row=1, col=1)
    fig.add_trace(go.Scatter(x=main_df.index, y=main_df['bb_lower'],
                             name='BB Lower', line=dict(color='gray', dash='dot'),
                             fill='tonexty', fillcolor='rgba(0,0,255,0.05)'), row=1, col=1)

# Normalized comparison
for pair, color in [('GBPUSD=X', 'orange'), ('USDJPY=X', 'green')]:
    series = dfs[pair]['Close']
    normalized = series / series.iloc[0]
    fig.add_trace(go.Scatter(x=series.index, y=normalized,
                             name=f'{pair} (Norm)', line=dict(color=color)), row=2, col=1)

fig.update_layout(title='Currency Exchange Rate Overview', height=600)
fig.show()
print('Price summary (EUR/USD):')
print(main_df['Close'].describe().round(4))

In [ ]:
# Cell 4: Returns Distribution Analysis
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

returns = main_df['daily_return'].dropna()

# Daily return histogram
axes[0, 0].hist(returns * 100, bins=80, color='steelblue', edgecolor='none', alpha=0.8)
axes[0, 0].axvline(0, color='red', linewidth=1.5, linestyle='--')
axes[0, 0].set_title('EUR/USD Daily Return Distribution')
axes[0, 0].set_xlabel('Daily Return (%)')

# QQ-like plot
from scipy.stats import probplot
probplot(returns, dist='norm', plot=axes[0, 1])
axes[0, 1].set_title('Normal Q-Q Plot')

# ACF of returns
lags = range(1, 41)
acf_vals = [pd.Series(returns.values).autocorr(lag) for lag in lags]
axes[1, 0].bar(list(lags), acf_vals, color='steelblue')
axes[1, 0].axhline(0, color='black', linewidth=0.8)
axes[1, 0].axhline(1.96 / np.sqrt(len(returns)), color='red', linestyle='--', alpha=0.7)
axes[1, 0].axhline(-1.96 / np.sqrt(len(returns)), color='red', linestyle='--', alpha=0.7)
axes[1, 0].set_title('Returns Autocorrelation (ACF)')
axes[1, 0].set_xlabel('Lag')

# ACF of squared returns (ARCH effects)
sq_returns = returns ** 2
acf_sq = [pd.Series(sq_returns.values).autocorr(lag) for lag in lags]
axes[1, 1].bar(list(lags), acf_sq, color='orange')
axes[1, 1].axhline(0, color='black', linewidth=0.8)
axes[1, 1].set_title('Squared Returns ACF (ARCH Effects)')
axes[1, 1].set_xlabel('Lag')

plt.suptitle('EUR/USD Returns Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Skewness: {returns.skew():.4f}')
print(f'Kurtosis: {returns.kurtosis():.4f}  (Normal=3.0)')
print(f'Annualized volatility: {returns.std() * np.sqrt(252) * 100:.2f}%')

In [ ]:
# Cell 5: Volatility Analysis
fig = go.Figure()
for vol_col, color in [('volatility_10', 'blue'), ('volatility_20', 'orange'), ('volatility_60', 'green')]:
    if vol_col in main_df.columns:
        label = f'{vol_col.split("_")[1]}-Day'
        fig.add_trace(go.Scatter(
            x=main_df.index,
            y=main_df[vol_col] * 100 * np.sqrt(252),
            name=f'Annualized Vol ({label})',
            line=dict(color=color)
        ))
fig.update_layout(
    title='EUR/USD Annualized Rolling Volatility',
    yaxis_title='Annualized Volatility (%)',
    height=400
)
fig.show()

# Volatility statistics
if 'volatility_20' in main_df.columns:
    ann_vol = main_df['volatility_20'] * 100 * np.sqrt(252)
    print(f'Mean annualized volatility (20-day): {ann_vol.mean():.2f}%')
    print(f'Max annualized volatility: {ann_vol.max():.2f}%')
    print(f'Min annualized volatility: {ann_vol.min():.2f}%')

In [ ]:
# Cell 6: Technical Indicators Overview
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# Panel 1: Price + Moving Averages
tail = main_df.tail(500)
axes[0].plot(tail.index, tail['Close'], label='Close', linewidth=1, color='royalblue')
for sma, color in [('sma_20', 'orange'), ('sma_50', 'red'), ('sma_200', 'purple')]:
    if sma in tail.columns:
        axes[0].plot(tail.index, tail[sma], label=sma.upper(), linewidth=1, linestyle='--', color=color)
axes[0].set_title('EUR/USD with Moving Averages (Last 500 days)')
axes[0].legend(loc='upper left', fontsize=8)
axes[0].grid(True, alpha=0.3)

# Panel 2: RSI
if 'rsi_14' in tail.columns:
    axes[1].plot(tail.index, tail['rsi_14'], color='purple', linewidth=1)
    axes[1].axhline(70, color='red', linestyle='--', alpha=0.7, label='Overbought (70)')
    axes[1].axhline(30, color='green', linestyle='--', alpha=0.7, label='Oversold (30)')
    axes[1].axhline(50, color='gray', linestyle='-', alpha=0.4)
    axes[1].set_title('RSI (14)')
    axes[1].set_ylim(0, 100)
    axes[1].legend(fontsize=8)
    axes[1].grid(True, alpha=0.3)

# Panel 3: MACD
if 'macd' in tail.columns:
    axes[2].plot(tail.index, tail['macd'], label='MACD', color='blue', linewidth=1)
    axes[2].plot(tail.index, tail['macd_signal'], label='Signal', color='orange', linewidth=1)
    axes[2].bar(tail.index, tail['macd_hist'], label='Histogram', color='gray', alpha=0.5, width=1)
    axes[2].axhline(0, color='black', linewidth=0.8)
    axes[2].set_title('MACD')
    axes[2].legend(fontsize=8)
    axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Cell 7: Feature Correlation Analysis
key_features = [
    'Close', 'daily_return', 'weekly_return', 'volatility_20',
    'rsi_14', 'macd', 'bb_position', 'atr_pct',
    'Close_lag_1', 'Close_lag_5', 'Close_lag_10',
    'volume_ratio', 'month_sin', 'dow_sin'
]
avail_features = [f for f in key_features if f in main_df.columns]
corr = main_df[avail_features].corr()

plt.figure(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, square=True, linewidths=0.5, vmin=-1, vmax=1,
            cbar_kws={'shrink': 0.8})
plt.title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# Top features correlated with Close
corrs_with_close = corr['Close'].abs().sort_values(ascending=False).head(10)
print('Top 10 features correlated with Close:')
print(corrs_with_close.round(3).to_string())

In [ ]:
# Cell 8: Stationarity Tests
from statsmodels.tsa.stattools import adfuller, kpss

def test_stationarity(series, name='Series'):
    series = series.dropna()
    # ADF Test
    adf_result = adfuller(series, autolag='AIC')
    adf_p = adf_result[1]
    # KPSS Test
    kpss_result = kpss(series, regression='c', nlags='auto')
    kpss_p = kpss_result[1]
    adf_stat = 'Stationary' if adf_p < 0.05 else 'Non-stationary'
    kpss_stat = 'Stationary' if kpss_p > 0.05 else 'Non-stationary'
    print(f'{name}:')
    print(f'  ADF p-value: {adf_p:.4f} -> {adf_stat}')
    print(f'  KPSS p-value: {kpss_p:.4f} -> {kpss_stat}')
    print()

print('=== Stationarity Tests ===')
test_stationarity(main_df['Close'], 'EUR/USD Close (levels)')
test_stationarity(main_df['daily_return'].dropna(), 'EUR/USD Daily Returns')
test_stationarity(main_df['Close'].diff().dropna(), 'EUR/USD First Difference')

print('Conclusion:')
print('  Raw prices: Non-stationary (unit root) -> need differencing for ARIMA')
print('  Daily returns: Stationary -> suitable for direct modeling')
print('  First difference: Stationary -> ARIMA(p,1,q) order appropriate')

In [ ]:
# Cell 9: Summary and Next Steps
train_df, val_df, test_df = train_test_split_time_series(main_df)
print('=' * 60)
print('         EDA SUMMARY')
print('=' * 60)
print(f'Total observations : {len(main_df)}')
print(f'Features engineered: {main_df.shape[1]}')
print(f'Date range         : {main_df.index[0].date()} to {main_df.index[-1].date()}')
print(f'Training set       : {len(train_df)} ({100*len(train_df)/len(main_df):.1f}%)')
print(f'Validation set     : {len(val_df)} ({100*len(val_df)/len(main_df):.1f}%)')
print(f'Test set           : {len(test_df)} ({100*len(test_df)/len(main_df):.1f}%)')
print()
print('KEY FINDINGS:')
print('  1. EUR/USD prices are non-stationary (unit root) -> use differencing')
print('  2. Returns show near-normality with slight fat tails (excess kurtosis)')
print('  3. Volatility clustering present -> GARCH or LSTM can model this')
print('  4. RSI and lag features are most predictive of future Close')
print('  5. MACD histogram shows clear trend-following signals')
print()
print('NEXT STEPS:')
print('  1. Fit baseline ARIMA model with auto_arima')
print('  2. Train LSTM with 60-day lookback window')
print('  3. Fit Prophet with yearly + weekly seasonality')
print('  4. Run Optuna for XGBoost hyperparameter search')
print('  5. Build ensemble model and evaluate on test set')
print('  6. Generate 30-day future forecast with confidence intervals')